# MovieLens Rating Prediction

Same 8 steps as before, but with **no config dictionary**. Every setting
(file paths, batch size, learning rate, embedding sizes...) is just a plain
Python variable, set once near the top of the step that needs it. Nothing is
tucked away in a `CONFIG["..."]["..."]` lookup, so you can see every value
and change it right where it's used.

The 8 steps:
1. Imports & Settings
2. Load Data
3. Preprocessing
4. Build Model
5. Loss & Metrics
6. Train
7. Evaluate
8. Predictions

**Before running:** make sure this notebook sits in the same folder as the
`data/` and `out/` folders from the zip (i.e. run it from inside `project/`),
or edit the file paths in Step 1.

## 1. Imports & Settings

All the basic settings live here as plain variables — no dictionary, no YAML
file. Want to change the batch size or a file path? Edit the variable right
here.

In [ ]:
import os
import json
import time

import numpy as np
import pandas as pd
import tensorflow as tf

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- File paths ---
TRAIN_CSV = "https://raw.githubusercontent.com/ManitaInn/MovieLen-Project/refs/heads/main/train.csv"
VAL_CSV = "https://raw.githubusercontent.com/ManitaInn/MovieLen-Project/refs/heads/main/val.csv"
TEST_CSV = "https://raw.githubusercontent.com/ManitaInn/MovieLen-Project/refs/heads/main/test.csv"

MODEL_DIR = "out/model_v1"
MODEL_WEIGHTS_PATH = "out/model_v1/model.weights.h5"
PREDICTIONS_CSV = "out/model_v1/predictions.csv"
TRAINING_HISTORY_JSON = "out/model_v1/history.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# --- Feature settings ---
GENRE_COLUMNS = [
    "genre_unknown", "genre_action", "genre_adventure", "genre_animation",
    "genre_children", "genre_comedy", "genre_crime", "genre_documentary",
    "genre_drama", "genre_fantasy", "genre_film-noir", "genre_horror",
    "genre_musical", "genre_mystery", "genre_romance", "genre_sci-fi",
    "genre_thriller", "genre_war", "genre_western",
]
NUM_GENRES = 19
NUM_USERS = 944      # user_id in [1, 943] -> index 0 reserved
NUM_MOVIES = 1683    # movie_id in [1, 1682] -> index 0 reserved
# num_occupations gets set after we see the training data (Step 3)

# --- Model settings ---
USER_EMBEDDING_DIM = 32
MOVIE_EMBEDDING_DIM = 32
OCCUPATION_EMBEDDING_DIM = 8
DENSE_LAYERS = [128, 64, 32]
DROPOUT_RATE = 0.2
L2_REG = 1e-6
RATING_MIN = 1.0
RATING_MAX = 5.0

# --- Training settings ---
BATCH_SIZE = 256
EPOCHS = 15
LEARNING_RATE = 0.001
SHUFFLE = True
EARLY_STOPPING_PATIENCE = 3

print("Settings loaded. Output dir:", MODEL_DIR)

Settings loaded. Output dir: out/model_v1


## 2. Load Data

Just read the three raw CSVs. Each row is one (user, movie, rating) pair,
plus the user's demographic info and the movie's genre flags already joined in.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)} | Test rows: {len(test_df)}")
train_df.head()

Train rows: 79619 | Val rows: 9942 | Test rows: 10439


,user_id,movie_id,ratings,age,gender,occupation,movie_title,release_date,genre_unknown,genre_action,...,genre_fantasy,genre_film-noir,genre_horror,genre_musical,genre_mystery,genre_romance,genre_sci-fi,genre_thriller,genre_war,genre_western
0,1,168,5,24,M,technician,Monty Python and the Holy Grail (1974),01-Jan-1974,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,172,5,24,M,technician,"Empire Strikes Back, The (1980)",01-Jan-1980,0,1,...,0,0,0,0,0,1,1,0,1,0
2,1,165,5,24,M,technician,Jean de Florette (1986),01-Jan-1986,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,156,4,24,M,technician,Reservoir Dogs (1992),01-Jan-1992,0,0,...,0,0,0,0,0,0,0,1,0,0
4,1,196,5,24,M,technician,Dead Poets Society (1989),01-Jan-1989,0,0,...,0,0,0,0,0,0,0,0,0,0


## 3. Preprocessing

Turns raw columns into model-ready numpy arrays:

- `user_id` / `movie_id` → used directly as embedding lookup indices (clipped to valid range).
- `age` → standardized (mean 0, std 1), fit on **train only**.
- `gender` → 1.0 for `M`, 0.0 for `F`.
- `occupation` → mapped to an integer index; index 0 is reserved for any
  occupation seen at inference time that wasn't seen during training.
- `release_date` (e.g. `01-Jan-1974`) → year extracted, then standardized like age.
- Genre columns → already a 19-dim multi-hot vector, used as-is.

The `Preprocessor` class below is `fit` once on the training set (so it learns
the age/year mean+std and the occupation vocabulary), then `transform` is applied
to train, val, and test — using the *same* fitted stats, which is what prevents
data leakage from val/test into training.

Then `MovieLensSequence` is a `tf.keras.utils.Sequence` that slices these arrays
into batches of `(inputs_dict, ratings)` for training/evaluation.

In [ ]:
def extract_release_year(release_date_series):
    """release_date looks like '01-Jan-1995' -> pull out just the year."""
    return pd.to_datetime(release_date_series, format="%d-%b-%Y", errors="coerce").dt.year


class Preprocessor:
    """Holds every fitted stat/encoder needed to go from raw CSV -> model inputs."""

    def __init__(self, genre_columns):
        self.genre_columns = genre_columns
        self.occupation_to_idx = {}
        self.age_mean, self.age_std = 0.0, 1.0
        self.year_mean, self.year_std = 0.0, 1.0
        self.num_users, self.num_movies = 0, 0

    def fit(self, df, num_users, num_movies):
        occupations = sorted(df["occupation"].unique())
        self.occupation_to_idx = {occ: i + 1 for i, occ in enumerate(occupations)}
        self.num_occupations = len(self.occupation_to_idx) + 1

        self.age_mean = float(df["age"].mean())
        self.age_std = float(df["age"].std() + 1e-8)

        years = extract_release_year(df["release_date"])
        self.year_mean = float(years.mean())
        self.year_std = float(years.std() + 1e-8)

        self.num_users = num_users
        self.num_movies = num_movies
        return self

    def transform(self, df, has_ratings):
        user_id = df["user_id"].astype(np.int32).clip(0, self.num_users - 1).to_numpy()
        movie_id = df["movie_id"].astype(np.int32).clip(0, self.num_movies - 1).to_numpy()

        age = ((df["age"].astype(np.float32) - self.age_mean) / self.age_std).to_numpy(dtype=np.float32)
        gender = (df["gender"] == "M").astype(np.float32).to_numpy()

        occupation = df["occupation"].map(self.occupation_to_idx).fillna(0).astype(np.int32).to_numpy()

        years = extract_release_year(df["release_date"]).fillna(self.year_mean)
        release_year = ((years.astype(np.float32) - self.year_mean) / self.year_std).to_numpy(dtype=np.float32)

        genres = df[self.genre_columns].astype(np.float32).to_numpy()

        out = {
            "user_id": user_id, "movie_id": movie_id, "age": age, "gender": gender,
            "occupation": occupation, "release_year": release_year, "genres": genres,
        }
        if has_ratings:
            out["ratings"] = df["ratings"].astype(np.float32).to_numpy()
        else:
            # keep original ids around so Step 8 can write them back out
            out["raw_user_id"] = df["user_id"].astype(np.int32).to_numpy()
            out["raw_movie_id"] = df["movie_id"].astype(np.int32).to_numpy()
        return out


preprocessor = Preprocessor(GENRE_COLUMNS).fit(train_df, NUM_USERS, NUM_MOVIES)

# Now that we've seen the training occupations, lock in the final count
NUM_OCCUPATIONS = preprocessor.num_occupations

train_arrays = preprocessor.transform(train_df, has_ratings=True)
val_arrays = preprocessor.transform(val_df, has_ratings=True)
test_arrays = preprocessor.transform(test_df, has_ratings=False)

print(f"Fitted occupations: {len(preprocessor.occupation_to_idx)} -> NUM_OCCUPATIONS={NUM_OCCUPATIONS}")

Fitted occupations: 21 -> NUM_OCCUPATIONS=22


In [ ]:
class MovieLensSequence(tf.keras.utils.Sequence):
    """Serves batches of preprocessed arrays as (inputs_dict, ratings) pairs."""

    def __init__(self, arrays, batch_size=256, shuffle=True, has_ratings=True, seed=42):
        super().__init__()
        self.arrays = arrays
        self.has_ratings = has_ratings
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n = len(arrays["user_id"])
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(self.n)
        if self.shuffle:
            self.rng.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(self.n / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        inputs = {
            "user_id": self.arrays["user_id"][batch_idx],
            "movie_id": self.arrays["movie_id"][batch_idx],
            "age": self.arrays["age"][batch_idx],
            "gender": self.arrays["gender"][batch_idx],
            "occupation": self.arrays["occupation"][batch_idx],
            "release_year": self.arrays["release_year"][batch_idx],
            "genres": self.arrays["genres"][batch_idx],
        }
        if self.has_ratings:
            return inputs, self.arrays["ratings"][batch_idx]
        return inputs

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.indices)


train_seq = MovieLensSequence(train_arrays, batch_size=BATCH_SIZE,
                               shuffle=SHUFFLE, has_ratings=True, seed=SEED)
val_seq = MovieLensSequence(val_arrays, batch_size=BATCH_SIZE,
                             shuffle=False, has_ratings=True, seed=SEED)
test_seq = MovieLensSequence(test_arrays, batch_size=BATCH_SIZE,
                              shuffle=False, has_ratings=False, seed=SEED)

print(f"Train batches: {len(train_seq)} | Val batches: {len(val_seq)} | Test batches: {len(test_seq)}")

Train batches: 312 | Val batches: 39 | Test batches: 41


## 4. Build Model

A "wide-ish" embedding network:

- Learns a latent vector for each `user_id` and `movie_id` (the classic
  collaborative-filtering signal), plus a smaller embedding for `occupation`.
- Concatenates those embeddings with the normalized side features
  (`age`, `gender`, `release_year`, 19-dim genre vector).
- Feeds everything through a small MLP (`128 → 64 → 32`, ReLU + dropout + L2).
- Output is squashed with a scaled sigmoid so every prediction lands in
  `[1, 5]` (smoother gradients than a hard clip, but still a guaranteed valid range).

`build_model()` just reads the settings variables from Step 1 directly —
no config object gets passed around.

In [ ]:
def build_model():
    reg = tf.keras.regularizers.l2(L2_REG)

    user_id = tf.keras.Input(shape=(), dtype=tf.int32, name="user_id")
    movie_id = tf.keras.Input(shape=(), dtype=tf.int32, name="movie_id")
    age = tf.keras.Input(shape=(), dtype=tf.float32, name="age")
    gender = tf.keras.Input(shape=(), dtype=tf.float32, name="gender")
    occupation = tf.keras.Input(shape=(), dtype=tf.int32, name="occupation")
    release_year = tf.keras.Input(shape=(), dtype=tf.float32, name="release_year")
    genres = tf.keras.Input(shape=(NUM_GENRES,), dtype=tf.float32, name="genres")

    user_emb = tf.keras.layers.Embedding(NUM_USERS, USER_EMBEDDING_DIM,
                                          embeddings_regularizer=reg, name="user_embedding")(user_id)
    movie_emb = tf.keras.layers.Embedding(NUM_MOVIES, MOVIE_EMBEDDING_DIM,
                                           embeddings_regularizer=reg, name="movie_embedding")(movie_id)
    occ_emb = tf.keras.layers.Embedding(NUM_OCCUPATIONS, OCCUPATION_EMBEDDING_DIM,
                                         embeddings_regularizer=reg, name="occupation_embedding")(occupation)

    side_features = tf.keras.layers.Concatenate(name="side_features")([
        tf.keras.layers.Reshape((1,))(age),
        tf.keras.layers.Reshape((1,))(gender),
        tf.keras.layers.Reshape((1,))(release_year),
        genres,
    ])

    x = tf.keras.layers.Concatenate(name="all_features")([user_emb, movie_emb, occ_emb, side_features])

    for i, units in enumerate(DENSE_LAYERS):
        x = tf.keras.layers.Dense(units, activation="relu", kernel_regularizer=reg, name=f"dense_{i}")(x)
        x = tf.keras.layers.Dropout(DROPOUT_RATE, name=f"dropout_{i}")(x)

    raw_output = tf.keras.layers.Dense(1, activation="linear", name="raw_rating")(x)
    output = tf.keras.layers.Activation("sigmoid", name="sigmoid")(raw_output)
    output = tf.keras.layers.Lambda(lambda t: t * (RATING_MAX - RATING_MIN) + RATING_MIN, name="rating")(output)
    output = tf.keras.layers.Reshape((), name="rating_scalar")(output)

    return tf.keras.Model(
        inputs={"user_id": user_id, "movie_id": movie_id, "age": age, "gender": gender,
                "occupation": occupation, "release_year": release_year, "genres": genres},
        outputs=output,
        name="movielens_rating_predictor",
    )


model = build_model()
model.summary()

Model: "movielens_rating_predictor"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ age (InputLayer)    │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ release_year        │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_id             │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_id            │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ occupation          │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_9 (Reshape) │ (None, 1)         │          0 │ age[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_10          │ (None, 1)         │          0 │ gender[0][0]      │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_11          │ (None, 1)         │          0 │ release_year[0][… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ genres (InputLayer) │ (None, 19)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 32)        │     30,208 │ user_id[0][0]     │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 32)        │     53,856 │ movie_id[0][0]    │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ occupation_embeddi… │ (None, 8)         │        176 │ occupation[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ side_features       │ (None, 22)        │          0 │ reshape_9[0][0],  │
│ (Concatenate)       │                   │            │ reshape_10[0][0], │
│                     │                   │            │ reshape_11[0][0], │
│                     │                   │            │ genres[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ all_features        │ (None, 94)        │          0 │ user_embedding[0… │
│ (Concatenate)       │                   │            │ movie_embedding[… │
│                     │                   │            │ occupation_embed… │
│                     │                   │            │ side_features[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_0 (Dense)     │ (None, 128)       │     12,160 │ all_features[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_0 (Dropout) │ (None, 128)       │          0 │ dense_0[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 106,769 (417.07 KB)

 Trainable params: 106,769 (417.07 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Loss & Metrics

- **Loss (MSE)**: rating prediction is regression, and the assignment is
  graded on MSE directly, so this optimizes exactly what we're evaluated on.
- **Metrics**: RMSE (same units as stars — easier to read than MSE) and MAE
  ("on average, how many stars off are we"), tracked alongside the loss just
  for human-readable sanity checks.

In [ ]:
def get_loss():
    return tf.keras.losses.MeanSquaredError()


class RootMeanSquaredError(tf.keras.metrics.Metric):
    def __init__(self, name="rmse", **kwargs):
        super().__init__(name=name, **kwargs)
        self.mse = tf.keras.metrics.MeanSquaredError()

    def update_state(self, y_true, y_pred, sample_weight=None):
        self.mse.update_state(y_true, y_pred, sample_weight=sample_weight)

    def result(self):
        return tf.sqrt(self.mse.result())

    def reset_state(self):
        self.mse.reset_state()


def get_metrics():
    return [RootMeanSquaredError(name="rmse"), tf.keras.metrics.MeanAbsoluteError(name="mae")]


loss_fn = get_loss()
metrics = get_metrics()
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
print("Loss:", loss_fn.name, "| Metrics:", [m.name for m in metrics])

Loss: mean_squared_error | Metrics: ['rmse', 'mae']


## 6. Train

A custom `tf.GradientTape` training loop (rather than `model.fit`) so it's
explicit what happens on every batch, and so we control early stopping
ourselves: after each epoch, if validation loss hasn't improved for
`EARLY_STOPPING_PATIENCE` epochs in a row, training stops and we keep the
best weights seen so far.

In [ ]:
def run_epoch(model, sequence, loss_fn, metrics, optimizer=None, training=True):
    for m in metrics:
        m.reset_state()
    epoch_loss = tf.keras.metrics.Mean()

    for batch_inputs, batch_ratings in sequence:
        if training:
            with tf.GradientTape() as tape:
                preds = model(batch_inputs, training=True)
                loss = loss_fn(batch_ratings, preds)
                loss += tf.add_n(model.losses) if model.losses else 0.0
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
        else:
            preds = model(batch_inputs, training=False)
            loss = loss_fn(batch_ratings, preds)

        epoch_loss.update_state(loss)
        for m in metrics:
            m.update_state(batch_ratings, preds)

    sequence.on_epoch_end()
    results = {"loss": float(epoch_loss.result().numpy())}
    for m in metrics:
        results[m.name] = float(m.result().numpy())
    return results


history = {"train": [], "val": []}
best_val_loss = np.inf
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_results = run_epoch(model, train_seq, loss_fn, metrics, optimizer, training=True)
    val_results = run_epoch(model, val_seq, loss_fn, metrics, optimizer=None, training=False)
    dt = time.time() - t0

    history["train"].append(train_results)
    history["val"].append(val_results)

    print(f"Epoch {epoch:02d}/{EPOCHS} ({dt:.1f}s) | "
          f"train_loss={train_results['loss']:.4f} rmse={train_results['rmse']:.4f} mae={train_results['mae']:.4f} | "
          f"val_loss={val_results['loss']:.4f} rmse={val_results['rmse']:.4f} mae={val_results['mae']:.4f}")

    if val_results["loss"] < best_val_loss - 1e-4:
        best_val_loss = val_results["loss"]
        patience_counter = 0
        model.save_weights(MODEL_WEIGHTS_PATH)
        print(f"  -> new best val_loss={best_val_loss:.4f}, weights saved.")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch} "
                  f"(no improvement for {EARLY_STOPPING_PATIENCE} epochs).")
            break

with open(TRAINING_HISTORY_JSON, "w") as f:
    json.dump(history, f, indent=2)

print(f"\nBest val_loss (MSE): {best_val_loss:.4f} | RMSE: {best_val_loss ** 0.5:.4f}")
print(f"Best weights saved to: {MODEL_WEIGHTS_PATH}")

Epoch 01/15 (39.7s) | train_loss=1.0322 rmse=1.0158 mae=0.8168 | val_loss=0.9471 rmse=0.9732 mae=0.7790
  -> new best val_loss=0.9471, weights saved.
Epoch 02/15 (39.5s) | train_loss=0.8749 rmse=0.9352 mae=0.7424 | val_loss=0.9237 rmse=0.9611 mae=0.7684
  -> new best val_loss=0.9237, weights saved.
Epoch 03/15 (39.7s) | train_loss=0.8377 rmse=0.9150 mae=0.7244 | val_loss=0.9104 rmse=0.9541 mae=0.7548
  -> new best val_loss=0.9104, weights saved.
Epoch 04/15 (40.2s) | train_loss=0.8122 rmse=0.9009 mae=0.7126 | val_loss=0.9017 rmse=0.9496 mae=0.7519
  -> new best val_loss=0.9017, weights saved.
Epoch 05/15 (41.6s) | train_loss=0.7918 rmse=0.8895 mae=0.7008 | val_loss=0.8958 rmse=0.9465 mae=0.7459
  -> new best val_loss=0.8958, weights saved.
Epoch 06/15 (41.3s) | train_loss=0.7635 rmse=0.8734 mae=0.6887 | val_loss=0.8965 rmse=0.9469 mae=0.7440
Epoch 07/15 (42.3s) | train_loss=0.7455 rmse=0.8630 mae=0.6798 | val_loss=0.8894 rmse=0.9431 mae=0.7414
  -> new best val_loss=0.8894, weights sav

## 7. Evaluate

Reload the best saved weights (the checkpoint from whichever epoch had the
lowest validation loss) and report MSE/RMSE/MAE on the validation split — the
split we never trained on directly. This is a sanity check before generating
test predictions in the next step.

In [ ]:
eval_model = build_model()
eval_model.load_weights(MODEL_WEIGHTS_PATH)

eval_metrics = get_metrics()
total_loss, n_batches = 0.0, 0

for batch_inputs, batch_ratings in val_seq:
    preds = eval_model(batch_inputs, training=False)
    total_loss += float(loss_fn(batch_ratings, preds).numpy())
    for m in eval_metrics:
        m.update_state(batch_ratings, preds)
    n_batches += 1

print(f"Validation MSE:  {total_loss / n_batches:.4f}")
for m in eval_metrics:
    print(f"Validation {m.name.upper()}: {float(m.result().numpy()):.4f}")

Validation MSE:  0.8894
Validation RMSE: 0.9431
Validation MAE: 0.7414


## 8. Predictions

Run the trained model on the test set (which has no `ratings` column), round
each prediction to the nearest integer star and clip to `[1, 5]` (ratings in
this dataset are always whole numbers), then write `predictions.csv` with the
columns `user_id, movie_id, ratings` — matching `sample_output.csv`'s format.

In [ ]:
all_preds = []
for i in range(len(test_seq)):
    batch_inputs = test_seq[i]
    preds = eval_model(batch_inputs, training=False).numpy()
    all_preds.append(preds)
predictions = np.concatenate(all_preds)

rounded = np.clip(np.round(predictions), 1, 5).astype(int)

out_df = pd.DataFrame({
    "user_id": test_arrays["raw_user_id"],
    "movie_id": test_arrays["raw_movie_id"],
    "ratings": rounded,
})
out_df.to_csv(PREDICTIONS_CSV, index=False)

print(f"Wrote {len(out_df)} predictions to {PREDICTIONS_CSV}")
out_df.head()

Wrote 10439 predictions to out/model_v1/predictions.csv


,user_id,movie_id,ratings
0,1,9,4
1,1,169,5
2,1,178,4
3,1,87,4
4,1,16,4
